# V5 grounded forecasting experiment
Use T4 or L4 without upgrading your account. This notebook does not touch v3 runs. The corpus is machine-validated pending human review; a pilot is not a completed research experiment. Run cells in order. Every stage reloads its paths after a reset. Final-test labels are never loaded here; use the separately gated final evaluator only after review and a frozen selection manifest. No keepalive or automatic account reauthentication is attempted.
Run folder: `MyDrive/FYP-model-runs/qwen35-2b-lora-v5-20260921T161000Z/`. For a distinct experiment, change this folder in every code cell before running; never repurpose this folder with different data/settings.

In [ ]:
import json, sys, shutil, subprocess
from pathlib import Path
from datetime import datetime, timezone
from importlib.metadata import version, PackageNotFoundError
import torch
from google.colab import drive
if not Path('/content/drive/MyDrive').is_dir():
    drive.mount('/content/drive')
RUN = Path('/content/drive/MyDrive/FYP-model-runs/qwen35-2b-lora-v5-20260921T161000Z')
RUN.mkdir(parents=True, exist_ok=True)
assert torch.cuda.is_available(), 'Select a GPU runtime; CPU training is not supported'
packages = {}
for name in ['torch', 'transformers', 'datasets', 'accelerate', 'peft', 'trl', 'pydantic', 'openai']:
    try:
        packages[name] = version(name)
    except PackageNotFoundError:
        packages[name] = 'missing'
info = {'utc': datetime.now(timezone.utc).isoformat(), 'python': sys.version, 'gpu': torch.cuda.get_device_name(0), 'vram_bytes': torch.cuda.get_device_properties(0).total_memory, 'cuda': torch.version.cuda, 'packages': packages, 'runtime_free_bytes': shutil.disk_usage('/content').free, 'drive_mount_free_bytes': shutil.disk_usage(RUN).free}
(RUN / ('inventory-' + datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ') + '.json')).write_text(json.dumps(info, indent=2))
print(json.dumps(info, indent=2))
print('Filesystem free space is not a verified Google account quota. No training has started.')


In [ ]:
import json, sys, subprocess
from pathlib import Path
from google.colab import drive
if not Path('/content/drive/MyDrive').is_dir():
    drive.mount('/content/drive')
RUN = Path('/content/drive/MyDrive/FYP-model-runs/qwen35-2b-lora-v5-20260921T161000Z')
PROJECT, FPY = Path('/content/slm-v5'), Path('/content/fpy')
RUN.mkdir(parents=True, exist_ok=True)
def command(*args):
    subprocess.run(list(map(str, args)), check=True)
if not (PROJECT / '.git').is_dir():
    command('git', 'clone', 'https://github.com/AbdullahUsman0/SLM-FineTuning-Testing.git', PROJECT)
lock = RUN / 'source-lock.json'
if not lock.exists():
    command('git', '-C', PROJECT, 'fetch', 'origin', 'experiment/v5-grounded-20260919')
    commit = subprocess.check_output(['git', '-C', str(PROJECT), 'rev-parse', 'FETCH_HEAD'], text=True).strip()
    with lock.open('x') as handle:
        json.dump({'repository_commit': commit, 'fpy_commit': '04d52c015d1e3ecdefe92b87116f209361509b4b', 'model': 'Qwen/Qwen3.5-2B', 'model_revision': '15852e8c16360a2fea060d615a32b45270f8a8fc', 'license': 'Apache-2.0'}, handle, indent=2)
pins = json.loads(lock.read_text())
for target, url, pin in [(PROJECT, 'https://github.com/AbdullahUsman0/SLM-FineTuning-Testing.git', pins['repository_commit']), (FPY, 'https://github.com/int-abd-5/fpy.git', pins['fpy_commit'])]:
    if not (target / '.git').is_dir():
        command('git', 'clone', url, target)
    assert not subprocess.check_output(['git', '-C', str(target), 'status', '--porcelain', '--untracked-files=no'], text=True).strip(), 'Preserve modified checkout before proceeding'
    command('git', '-C', target, 'fetch', 'origin', pin)
    command('git', '-C', target, 'checkout', '--detach', pin)
command(sys.executable, '-m', 'pip', 'install', '-r', PROJECT / 'training/requirements.txt')
command(sys.executable, '-m', 'pip', 'install', '-e', FPY)
print(json.dumps(pins, indent=2))
print('If Colab asks to restart after package installation, restart and continue with the next cell.')


In [ ]:
import json, hashlib
from pathlib import Path
from huggingface_hub import hf_hub_download
from google.colab import drive
if not Path('/content/drive/MyDrive').is_dir():
    drive.mount('/content/drive')
RUN = Path('/content/drive/MyDrive/FYP-model-runs/qwen35-2b-lora-v5-20260921T161000Z')
PROJECT = Path('/content/slm-v5')
pins = json.loads((RUN / 'source-lock.json').read_text())
provenance = json.loads((PROJECT / 'training/v5-model-provenance.json').read_text())
assert provenance['revision'] == pins['model_revision'] and provenance['model_id'] == pins['model']
verified = []
for artifact in provenance['weights']:
    path = Path(hf_hub_download(repo_id=pins['model'], filename=artifact['file'], revision=pins['model_revision']))
    digest = hashlib.sha256()
    with path.open('rb') as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b''):
            digest.update(chunk)
    assert path.stat().st_size == artifact['size_bytes'] and digest.hexdigest() == artifact['sha256'], 'Base weights failed publisher hash verification'
    verified.append(artifact)
result = {'model': pins['model'], 'revision': pins['model_revision'], 'license': provenance['license'], 'verified_downloads': verified}
(RUN / 'base-weights-verified.json').write_text(json.dumps(result, indent=2))
print(json.dumps(result, indent=2))

In [ ]:
import sys, json, hashlib, shutil, subprocess, random
from pathlib import Path
from collections.abc import Mapping
from transformers import AutoTokenizer
from google.colab import drive
if not Path('/content/drive/MyDrive').is_dir():
    drive.mount('/content/drive')
RUN = Path('/content/drive/MyDrive/FYP-model-runs/qwen35-2b-lora-v5-20260921T161000Z')
PROJECT = Path('/content/slm-v5')
CORPUS = PROJECT / 'corpus-v5/v5-20260919-r1'
subprocess.run([sys.executable, str(PROJECT / 'scripts/verify-corpus-v5.py'), '--output', str(CORPUS)], cwd=PROJECT, check=True)
inputs = RUN / 'inputs'
inputs.mkdir(exist_ok=True)
for name in ['train', 'validation']:
    source, dest = CORPUS / 'sft' / (name + '.jsonl'), inputs / (name + '.jsonl')
    if not dest.exists():
        shutil.copy2(source, dest)
    assert hashlib.sha256(dest.read_bytes()).digest() == hashlib.sha256(source.read_bytes()).digest(), 'Frozen input hash mismatch'
manifest_hash = hashlib.sha256((CORPUS / 'manifest.json').read_bytes()).hexdigest()
pins = json.loads((RUN / 'source-lock.json').read_text())
tokenizer = AutoTokenizer.from_pretrained(pins['model'], revision=pins['model_revision'])
report = {'corpus_manifest_sha256': manifest_hash, 'splits': {}}
for name, count in [('train', 32), ('validation', 16)]:
    rows = [json.loads(line) for line in (inputs / (name + '.jsonl')).read_text().splitlines() if line.strip()]
    lengths = []
    for row in rows:
        messages = row.get('messages') or [*row['prompt'], *row['completion']]
        tokenized = tokenizer.apply_chat_template(messages, tokenize=True, add_generation_prompt=False, truncation=False, enable_thinking=False)
        if isinstance(tokenized, Mapping):
            tokenized = tokenized['input_ids']
        if tokenized and isinstance(tokenized[0], list):
            tokenized = tokenized[0]
        lengths.append(len(tokenized))
    report['splits'][name] = {'examples': len(rows), 'minimum_tokens': min(lengths), 'maximum_tokens': max(lengths), 'mean_tokens': sum(lengths) / len(lengths)}
    random.Random(42).shuffle(rows)
    pilot = inputs / ('pilot-' + name + '.jsonl')
    data = ''.join(json.dumps(row, ensure_ascii=False) + '\n' for row in rows[:count]).encode('utf-8')
    if pilot.exists():
        assert pilot.read_bytes() == data, 'Pilot inputs changed'
    else:
        pilot.write_bytes(data)
maximum = max(value['maximum_tokens'] for value in report['splits'].values())
report['max_length'] = ((maximum + 127) // 128) * 128
length_path = RUN / 'length-preflight.json'
if length_path.exists():
    assert json.loads(length_path.read_text()) == report, 'Length preflight changed'
else:
    length_path.write_text(json.dumps(report, indent=2))
print(json.dumps(report, indent=2))
print('Tracked development artifacts verified. Sealed final hashes remain local and are not checked in Colab. Do not truncate. Pilot OOM requires a separately documented new configuration/run, not silent settings changes.')

In [ ]:
import json, sys, subprocess, time
from pathlib import Path
from google.colab import drive
if not Path('/content/drive/MyDrive').is_dir():
    drive.mount('/content/drive')
RUN = Path('/content/drive/MyDrive/FYP-model-runs/qwen35-2b-lora-v5-20260921T161000Z')
PROJECT = Path('/content/slm-v5')
pins = json.loads((RUN / 'source-lock.json').read_text())
preflight = json.loads((RUN / 'length-preflight.json').read_text())
args = [sys.executable, 'training/train_lora.py', '--model', pins['model'], '--revision', pins['model_revision'], '--train', str(RUN / 'inputs/pilot-train.jsonl'), '--validation', str(RUN / 'inputs/pilot-validation.jsonl'), '--output', str(RUN / 'pilot'), '--epochs', '1', '--learning-rate', '5e-5', '--early-stopping-patience', '0', '--max-length', str(preflight['max_length']), '--resume-from-checkpoint', 'auto']
start = time.monotonic()
with (RUN / 'pilot-console.log').open('a') as log:
    result = subprocess.run(args, cwd=PROJECT, stdout=log, stderr=subprocess.STDOUT)
print('Pilot elapsed seconds:', time.monotonic() - start, 'exit:', result.returncode)
result.check_returncode()
print('Pilot only. Inspect pilot-console.log, raw development outputs and human review before full training.')


In [ ]:
import sys, json, subprocess
from pathlib import Path
from google.colab import drive
if not Path('/content/drive/MyDrive').is_dir():
    drive.mount('/content/drive')
RUN = Path('/content/drive/MyDrive/FYP-model-runs/qwen35-2b-lora-v5-20260921T161000Z')
PROJECT = Path('/content/slm-v5')
pins = json.loads((RUN / 'source-lock.json').read_text())
for candidate in ['base', 'lora']:
    output = RUN / ('pilot-' + candidate + '-smoke.json')
    if output.exists():
        print('Existing report preserved:', output)
        continue
    args = [sys.executable, 'scripts/evaluate-v5.py', '--provider', candidate, '--model', pins['model'], '--revision', pins['model_revision'], '--cases', 'corpus-v5/v5-20260919-r1/splits/smoke.jsonl', '--output', str(output), '--device', 'cuda', '--dtype', 'float16', '--max-new-tokens', '3072']
    if candidate == 'lora':
        args += ['--adapter', str(RUN / 'pilot/best-adapter')]
    subprocess.run(args, cwd=PROJECT, check=True)
print('Inspect only development smoke outputs, not sealed final cases. Re-estimate full-run GPU time from pilot logs.')


In [ ]:
import sys, json, subprocess
from pathlib import Path
from google.colab import drive
if not Path('/content/drive/MyDrive').is_dir():
    drive.mount('/content/drive')
RUN = Path('/content/drive/MyDrive/FYP-model-runs/qwen35-2b-lora-v5-20260921T161000Z')
PROJECT = Path('/content/slm-v5')
pins = json.loads((RUN / 'source-lock.json').read_text())
preflight = json.loads((RUN / 'length-preflight.json').read_text())
review = json.loads((RUN / 'human-review-approval.json').read_text())
assert review['corpus_manifest_sha256'] == preflight['corpus_manifest_sha256']
assert review['train_sample_approved'] is True and review['validation_approved'] is True and review['reviewer']
assert review['pilot_behavior_approved'] is True and review['compute_estimate_approved'] is True
args = [sys.executable, 'training/train_lora.py', '--model', pins['model'], '--revision', pins['model_revision'], '--train', str(RUN / 'inputs/train.jsonl'), '--validation', str(RUN / 'inputs/validation.jsonl'), '--output', str(RUN / 'full'), '--epochs', '2', '--learning-rate', '5e-5', '--early-stopping-patience', '0', '--max-length', str(preflight['max_length']), '--resume-from-checkpoint', 'auto']
with (RUN / 'full-console.log').open('a') as log:
    result = subprocess.run(args, cwd=PROJECT, stdout=log, stderr=subprocess.STDOUT)
result.check_returncode()
print('Epoch checkpoints preserved. best-adapter is loss-best, NOT the behavioral winner. Compare each epoch on validation before freezing selection.')


In [ ]:
import os, sys, json, subprocess
from pathlib import Path
from google.colab import userdata, drive
if not Path('/content/drive/MyDrive').is_dir():
    drive.mount('/content/drive')
RUN = Path('/content/drive/MyDrive/FYP-model-runs/qwen35-2b-lora-v5-20260921T161000Z')
PROJECT = Path('/content/slm-v5')
assert os.environ.get('V5_OPENAI_COST_APPROVED') == 'yes', 'Review current API rates and approve the validation-call budget before running this optional cell'
os.environ['OPENAI_API_KEY'] = userdata.get('OPENAI_API_KEY')
os.environ['OPENAI_MODEL'] = os.environ.get('OPENAI_MODEL', 'gpt-5-mini-2025-08-07')
try:
    subprocess.run([sys.executable, 'scripts/evaluate-v5.py', '--provider', 'openai', '--model', os.environ['OPENAI_MODEL'], '--cases', 'corpus-v5/v5-20260919-r1/splits/validation.jsonl', '--output', str(RUN / 'openai-validation.json')], cwd=PROJECT, check=True)
finally:
    os.environ.pop('OPENAI_API_KEY', None)
print('If the snapshot is unavailable, record that outcome and explicitly choose an accessible exact model ID; do not substitute historical results.')


## Selection and final evaluation
Use `scripts/compare-v5.py` on matched validation reports, keeping every epoch checkpoint. Primary selection metric is non-intent exact slot F1, with no forbidden-inference increase and at most 0.01 absolute JSON/correction regression. Bootstrap uses 10,000 paired scenario-cluster resamples, seed 42. Freeze the exact selected adapter, prompt/code/model/corpus hashes and review approval before opening final labels. The `evaluate-v5.py --split final` gate must reject missing approval or repeated evaluations. Final results are **not measured** until these conditions hold.
On runtime loss: reopen this notebook, select GPU, rerun inventory and pinned-source stages, then the immutable input verification and the appropriate pilot/full training cell. Never replace data, alter optimizer settings or select a partial checkpoint to make resume succeed. Reauthentication and Colab quota require user intervention. Keep all v3 artifacts unchanged.